In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch07-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "07-03-matched-molecular-pairs.py"

# 芳香環上的氫換成氯，活性會怎麼變？

第一次執行時，DeepChem 會下載 BACE 資料。要試做小練習，可把程式接在分析儲存格的末端，再執行分析格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
from collections import defaultdict
import os
import deepchem as dc
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdDepictor, Draw

# BACE-1：1,513 筆實測 pIC50。原始論文 DOI 10.1021/acs.jcim.6b00290。
tasks, (data,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
measurements = defaultdict(list)
for smi, y, w in zip(data.ids, data.y[:, 0], data.w[:, 0]):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None and w > 0 and np.isfinite(y):
        measurements[Chem.MolToSmiles(mol)].append(float(y))
activity = {s: float(np.median(values)) for s, values in measurements.items()}

records = []
for cl_smi, cl_value in activity.items():
    mol = Chem.MolFromSmiles(cl_smi)
    seen = set()
    for atom in mol.GetAtoms():
        if atom.GetSymbol() != "Cl" or atom.GetDegree() != 1:
            continue
        carbon = atom.GetNeighbors()[0]
        if carbon.GetSymbol() != "C" or not carbon.GetIsAromatic():
            continue
        parent = Chem.RWMol(mol)
        parent.RemoveAtom(atom.GetIdx())
        Chem.SanitizeMol(parent)
        h_smi = Chem.MolToSmiles(parent)
        if h_smi in activity and h_smi not in seen:
            records.append((h_smi, cl_smi, activity[h_smi], cl_value, atom.GetIdx()))
            seen.add(h_smi)
pairs = pd.DataFrame(records, columns=["H", "Cl", "pIC50_H", "pIC50_Cl", "cl_idx"])
pairs["delta"] = pairs["pIC50_Cl"] - pairs["pIC50_H"]
pairs["ratio"] = 10 ** pairs["delta"]  # IC50_H / IC50_Cl
print("unique structures / pairs / |delta| >= 1:",
      len(activity), len(pairs), (pairs.delta.abs() >= 1).sum())
examples = pairs.loc[[pairs.delta.idxmax(), pairs.delta.idxmin()]]
print(examples.round(4).to_string(index=False))

# 由同一個含氯分子的座標產生左右兩張結構圖，讓取代位置對齊。
molecules, legends, highlights = [], [], []
for label, row in zip(["A", "B"], examples.itertuples()):
    derivative = Chem.MolFromSmiles(row.Cl)
    rdDepictor.Compute2DCoords(derivative, sampleSeed=42)
    idx = row.cl_idx
    site = derivative.GetAtomWithIdx(idx).GetNeighbors()[0].GetIdx()
    parent = Chem.RWMol(derivative)
    parent.RemoveAtom(idx)
    Chem.SanitizeMol(parent)
    assert Chem.MolToSmiles(parent) == row.H
    molecules.extend([parent, derivative])
    highlights.extend([[site - int(site > idx)], [site, idx]])
    legends.extend([f"{label}   Ar-H    pIC50 = {row.pIC50_H:.2f}",
                    f"{label}   Ar-Cl    pIC50 = {row.pIC50_Cl:.2f}"])
options = Draw.MolDrawOptions()
options.legendFontSize = 24
grid = Draw.MolsToGridImage(molecules, molsPerRow=2, subImgSize=(490, 300),
                           legends=legends, highlightAtomLists=highlights,
                           drawOptions=options, returnPNG=True)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
(out / "07-03-matched-molecular-pairs.png").write_bytes(grid)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "07-03-matched-molecular-pairs.png")))